<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 05 · Unsupervised Overview
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
Chapter 7 is intentionally light. The goal is to make clustering and PCA feel
familiar enough that the reader can recognize when they are useful.

## Setup
We start by fixing the book root so the helper import and the notebook runtime
stay predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Churn Helper
The helper import is separated out first so the clustering cell can focus on
standardization, dimensionality reduction, and group summaries.


In [ ]:
from pathlib import Path
import importlib.util
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


## Reduce the Features and Find Clusters
This notebook stays intentionally light. One code cell is enough to standardize
the numeric columns, reduce them with PCA, cluster the points, and inspect the
resulting group averages.


In [ ]:
df = churn_report.load_customers()
numeric = df[churn_report.NUMERIC_COLUMNS]

scaled = StandardScaler().fit_transform(numeric)
pca = PCA(n_components=2, random_state=0)
reduced = pca.fit_transform(scaled)
labels = KMeans(n_clusters=2, random_state=0, n_init=10).fit_predict(scaled)

summary = pd.DataFrame(
    reduced,
    columns=['pc1', 'pc2'],
    index=df.index,
)
summary['cluster'] = labels
summary['customer_id'] = df[churn_report.ID_COLUMN].values

print('explained variance ratio =', pca.explained_variance_ratio_)
print()
print(summary.to_string(index=False))
print()
cluster_summary = (
    df.assign(cluster=labels)
    .groupby('cluster')[churn_report.NUMERIC_COLUMNS]
    .mean()
    .round(2)
)
print(cluster_summary.to_string())


### Where We Are Heading Next
Chapter 8 turns the same care and skepticism into a capstone evaluation report.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">